# Explore Garmin data, prepocess it & export to csv

## Sources:
- [Analysis of Running Activities from Garmin Watch Using Python](https://towardsdatascience.com/analysis-of-runing-activities-from-garmin-watch-using-python-99609f83314e)

## ToDo:
- decide what to do when multiple activies in same day

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path

# import 3rd-party modules
import pandas as pd
import numpy as np

# import local modules

## Define functions

In [161]:
# Create Function to explore the dataframes
def explore(df: pd.DataFrame) -> None:
    """
    Function to print general information about the dataframe
    """
    print("********** 1. General info of data **********")
    print(df.info())
    
    print("\n********** 2. Shape of data **********")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")
    
    print("\n********** 3. Number of missing values per column **********")
    print(df.isnull().sum())
    
    print("\n********** 4. Number of duplicated values **********")
    print(df.duplicated().sum())
    
    print("\n********** 5. Number of unique values per column (NaN non included) **********")
    print(df.nunique())
    
    print("\n********** 6. Statistical info of each column **********")
    # print(df.describe(include='all').T)
    return df.describe(include='all', datetime_is_numeric=True).T

def convert_strings_to_duration(str_series):
    """
    Function to convert series of strings to durations; workaround when multiple formats in series
    """
    return pd.to_timedelta(pd.to_datetime(str_series).dt.strftime("%H:%M:%S.%f"))

def convert_durations_to_minutes(durations_series):
    # durations_series.dt.hour*60 + durations_series.dt.minute + durations_series.dt.second/60
    return durations_series.dt.total_seconds()/60

def normalize_between_range(xs, a, b):
    """
    Function to normalize a series between range [a,b]
    """
    min_x = np.min(xs)
    max_x = np.max(xs)
    return (b - a) * (xs - min_x)/(max_x - min_x) + a

## Read data

In [13]:
# set csv path
df_path = Path("assets/data/garmin_data/Activities_20210304_20220818.csv")

# read csv into dataframe
df = pd.read_csv(df_path, parse_dates=True)

## Explore data (Exploratory data analysis)

In [4]:
df.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Min Resp,Max Resp,Stress Change,Stress Start,Stress End,Avg Stress,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Cycling,2022-08-18 18:16:52,False,Schaarbeek Cycling,12.50,322,00:43:06,118,168,2.1,...,--,--,--,--,--,--,00:41:59,03:09:07,43,108
1,Cycling,2022-08-18 08:52:03,False,Ukkel Cycling,9.58,247,00:34:42,116,138,1.5,...,--,--,--,--,--,--,00:33:40,00:40:25,-12,115
2,Strength Training,2022-08-17 19:28:43,False,Strength,0.00,876,02:49:03,109,147,2.1,...,--,--,--,--,--,--,01:08:18,12:41:38,--,--
3,Cycling,2022-08-17 18:40:09,False,Vilvoorde Cycling,7.68,197,00:25:27,118,147,1.5,...,--,--,--,--,--,--,00:24:53,00:27:36,10,87
4,Cycling,2022-08-16 19:16:46,False,Kraainem Cycling,13.28,365,00:55:59,112,136,1.1,...,--,--,--,--,--,--,00:49:36,01:46:01,11,64


In [7]:
explore(df)

********** 1. General info of data **********
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 50 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Activity Type             765 non-null    object 
 1   Date                      765 non-null    object 
 2   Favorite                  765 non-null    bool   
 3   Title                     765 non-null    object 
 4   Distance                  765 non-null    object 
 5   Calories                  765 non-null    object 
 6   Time                      765 non-null    object 
 7   Avg HR                    765 non-null    int64  
 8   Max HR                    765 non-null    int64  
 9   Aerobic TE                765 non-null    object 
 10  Avg Run Cadence           765 non-null    object 
 11  Max Run Cadence           765 non-null    object 
 12  Avg Speed                 765 non-null    object 
 13  Max Speed          

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Activity Type,765,9,Cycling,371,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,765,765,2021-03-20 23:58:35,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Favorite,765,1,False,765,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,765,36,Vilvoorde Cycling,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Distance,765,414,0.00,253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Calories,765,363,--,172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time,765,566,00:15:34,161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Avg HR,765.0,NaN,NaN,NaN,105.294118,27.685094,44.0,98.0,114.0,122.0,165.0
Max HR,765.0,NaN,NaN,NaN,136.592157,31.842355,58.0,124.0,144.0,157.0,199.0
Aerobic TE,765,48,0.0,169,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Clean data

In [14]:
# select columns to keep
selected_cols = [
    "Activity Type",
    "Date",
    "Distance",
    "Calories",	"Time", "Avg HR", "Max HR", "Aerobic TE", "Avg Run Cadence",
    "Max Run Cadence", "Avg Speed", "Max Speed", "Total Ascent", "Total Descent", "Avg Stride Length", 
    "Best Lap Time", "Number of Laps", "Max Temp", "Moving Time", "Elapsed Time", "Min Elevation", "Max Elevation"
    ]

df = df[selected_cols]

### ToDo: find better solution to parse duration strings
ideas: 
- coerce error to convert error to nan and then fill na with other format
- consolidate strings in 1 format

In [15]:
# convert concerned cols to date & duration time
# df['Date'] = pd.to_datetime(pd.to_datetime(df['Date']).dt.date)
df['Date'] = pd.to_datetime(df.loc[:, 'Date'])
df["Date_yy-mm-dd"] = pd.to_datetime(df.loc[:, 'Date']).dt.date
df['Time'] = convert_strings_to_duration(df.loc[:, 'Time'])
df['Elapsed Time'] = convert_strings_to_duration(df.loc[:, 'Elapsed Time'])
df['Best Lap Time'] = df.loc[:, 'Best Lap Time'].apply(lambda x: f"00:{x}" if x.count(":") == 1 else x) # need to add hh:
df['Best Lap Time'] = pd.to_timedelta(df.loc[:, 'Best Lap Time'])
df['Moving Time'] = convert_strings_to_duration(df.loc[:, 'Moving Time'])

# convert durations cols to number of minutes
duration_cols = df.select_dtypes(include=["timedelta64[ns]"]).columns

for duration_col in duration_cols:
    df[duration_col] = convert_durations_to_minutes(df[duration_col])

In [17]:
# replace dummy value "--" by 0 (toDo: check impact of this replacement for other activities)
df.loc[:, "Calories"].replace({'--':'0'}, inplace=True)

# replace dummy value "--" by nan
df.replace({'--':np.nan}, inplace=True)

# # remove comma (to indicate thousands)
# df["Calories"] = df.loc[:, "Calories"].str.replace(',', "")
# df["Distance"] = df.loc[:, "Distance"].str.replace(',', "")

In [19]:
# convert object cols to float
## select only object cols
object_cols = df.select_dtypes(include='object').columns.to_list()

## remove title from object cols
object_cols.remove("Activity Type")
object_cols.remove("Avg Speed") # toDo: convert swimming pace (e.g. 1:58 /100m to kph)
object_cols.remove("Max Speed") # toDo: convert swimming pace (e.g. 1:58 /100m to kph)
object_cols.remove("Date_yy-mm-dd")

## remove comma (to indicate thousands)
for object_col in object_cols:
    df[object_col] = df[object_col].str.replace(',', "")

## convert object cols to float
df[object_cols] = df[object_cols].astype("float")

## inspect cols info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Activity Type      765 non-null    object        
 1   Date               765 non-null    datetime64[ns]
 2   Distance           765 non-null    float64       
 3   Calories           765 non-null    float64       
 4   Time               765 non-null    float64       
 5   Avg HR             765 non-null    int64         
 6   Max HR             765 non-null    int64         
 7   Aerobic TE         764 non-null    float64       
 8   Avg Run Cadence    764 non-null    float64       
 9   Max Run Cadence    764 non-null    float64       
 10  Avg Speed          513 non-null    object        
 11  Max Speed          513 non-null    object        
 12  Total Ascent       481 non-null    float64       
 13  Total Descent      480 non-null    float64       
 14  Avg Stride

In [20]:
df.head()

,Activity Type,Date,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,Avg Run Cadence,Max Run Cadence,...,Total Descent,Avg Stride Length,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation,Date_yy-mm-dd
0,Cycling,2022-08-18 18:16:52,12.50,322.0,43.100000,118,168,2.1,0.0,0.0,...,106.0,0.0,8.744267,3,0.0,41.983333,189.116667,43.0,108.0,2022-08-18
1,Cycling,2022-08-18 08:52:03,9.58,247.0,34.700000,116,138,1.5,0.0,0.0,...,69.0,0.0,13.636750,2,0.0,33.666667,40.416667,-12.0,115.0,2022-08-18
2,Strength Training,2022-08-17 19:28:43,0.00,876.0,169.050000,109,147,2.1,0.0,0.0,...,NaN,0.0,169.051500,1,0.0,68.300000,761.633333,NaN,NaN,2022-08-17
3,Cycling,2022-08-17 18:40:09,7.68,197.0,25.450000,118,147,1.5,0.0,0.0,...,105.0,0.0,8.838833,2,0.0,24.883333,27.600000,10.0,87.0,2022-08-17
4,Cycling,2022-08-16 19:16:46,13.28,365.0,55.983333,112,136,1.1,0.0,0.0,...,94.0,0.0,13.459233,3,0.0,49.600000,106.016667,11.0,64.0,2022-08-16


In [21]:
df[["Date"]].nunique()

Date    765
dtype: int64

## Group activities by date

In [22]:
# select columns
selected_cols = ["Date_yy-mm-dd", "Activity Type", "Distance", "Calories", "Time", "Avg HR", "Max HR"]

# group dataframe by date and activity Type
grouped_df_group = df[selected_cols].groupby(["Date_yy-mm-dd", "Activity Type"])
# grouped_by_date_df_group.groups

# aggregate values
grouped_df = grouped_df_group.agg({
    'Distance' : 'sum', 
    'Calories' : 'sum', 
    'Time' : 'sum', 
    'Avg HR' : 'mean',
    'Max HR' : 'mean'
    })

# reset index to have Date_yy-mm-dd col as regular column
grouped_df.reset_index(inplace=True)

# now, we can convert Date_yy-mm-dd column to datetime
grouped_df["Date_yy-mm-dd"] = pd.to_datetime(grouped_df["Date_yy-mm-dd"])
grouped_df.head()

,Date_yy-mm-dd,Activity Type,Distance,Calories,Time,Avg HR,Max HR
0,2021-03-04,Strength Training,0.0,411.0,62.933333,119.0,153.0
1,2021-03-05,Breathwork,0.0,0.0,5.841667,59.0,73.0
2,2021-03-05,Pilates,0.0,7.0,5.665000,75.0,92.0
3,2021-03-05,Yoga,0.0,1.0,2.826667,75.0,86.0
4,2021-03-07,Breathwork,0.0,0.0,15.566667,63.0,84.0


In [153]:
# get date range of dataframe
# date_start = df[["Date"]].min().dt.date.values[0]
# date_end = df[["Date"]].max().dt.date.values[0]
# print(date_start)
# print(date_end)

# get starting year and ending year
date_start = df[["Date"]].min().dt.year.values[0]
date_end = df[["Date"]].max().dt.year.values[0]
date_start = str(date_start) + "-01-01"
date_end = str(date_end) + "-12-31"
print(date_start)
print(date_end)

# get days between range date and make a dataframe from these days
range_date = pd.date_range(start=date_start, end=date_end, freq ='D')
range_date_df = pd.DataFrame(range_date, columns = ['Date_yy-mm-dd'])

# add date related columns to dataframe
range_date_df["Year"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.year
range_date_df["Month"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.month
range_date_df["Month_name"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.month_name()
range_date_df["Day"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.day
range_date_df["Day_name"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.day_name()
range_date_df["Day_of_week"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.day_of_week
range_date_df["Week_of_month"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).apply(lambda x: (x.day - 1)//7 + 1)

# add column: number of days in month
range_date_df["Days_in_month"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.days_in_month
range_date_df

2021-01-01
2022-12-31


,Date_yy-mm-dd,Year,Month,Month_name,Day,Day_name,Day_of_week,Week_of_month,Days_in_month
0,2021-01-01,2021,1,January,1,Friday,4,1,31
1,2021-01-02,2021,1,January,2,Saturday,5,1,31
2,2021-01-03,2021,1,January,3,Sunday,6,1,31
3,2021-01-04,2021,1,January,4,Monday,0,1,31
4,2021-01-05,2021,1,January,5,Tuesday,1,1,31
...,...,...,...,...,...,...,...,...,...
725,2022-12-27,2022,12,December,27,Tuesday,1,4,31
726,2022-12-28,2022,12,December,28,Wednesday,2,4,31
727,2022-12-29,2022,12,December,29,Thursday,3,5,31
728,2022-12-30,2022,12,December,30,Friday,4,5,31


In [154]:
# merge dataframe with all days from date range and dataframe with aggregated activities values
everyday_grouped_df = pd.merge(range_date_df, grouped_df, how="left", on="Date_yy-mm-dd")
everyday_grouped_df.head()

,Date_yy-mm-dd,Year,Month,Month_name,Day,Day_name,Day_of_week,Week_of_month,Days_in_month,Activity Type,Distance,Calories,Time,Avg HR,Max HR
0,2021-01-01,2021,1,January,1,Friday,4,1,31,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-02,2021,1,January,2,Saturday,5,1,31,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-03,2021,1,January,3,Sunday,6,1,31,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-04,2021,1,January,4,Monday,0,1,31,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-05,2021,1,January,5,Tuesday,1,1,31,NaN,NaN,NaN,NaN,NaN,NaN


In [155]:
# # get number of days per month and year
# nb_days_per_month_df = everyday_grouped_df[["Year", "Month", "Day"]].groupby(["Year", "Month"]).nunique()

# get number of activities per day
nb_activities_per_day = everyday_grouped_df[["Date_yy-mm-dd"]].value_counts(sort=False)
nb_activities_per_day = nb_activities_per_day.reset_index().rename(columns={0:"Total_nb_activities_per_day"})

# everyday_grouped_df[["Year", "Month", "Day"]].groupby(["Year", "Month"]).agg(["nunique", "count"])

# add number of activities per day to dataframe
everyday_grouped_df = pd.merge(everyday_grouped_df, nb_activities_per_day, how="left", on="Date_yy-mm-dd")
everyday_grouped_df.head()

,Date_yy-mm-dd,Year,Month,Month_name,Day,Day_name,Day_of_week,Week_of_month,Days_in_month,Activity Type,Distance,Calories,Time,Avg HR,Max HR,Total_nb_activities_per_day
0,2021-01-01,2021,1,January,1,Friday,4,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2021-01-02,2021,1,January,2,Saturday,5,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2021-01-03,2021,1,January,3,Sunday,6,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2021-01-04,2021,1,January,4,Monday,0,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2021-01-05,2021,1,January,5,Tuesday,1,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1


In [249]:
year_min = everyday_grouped_df.Year.min()
nb_months = 12
nb_cols_per_year = 4
nb_rows_per_year = nb_months // nb_cols_per_year
nb_days_in_week = 7


year_grid_width = 1 * nb_cols_per_year * nb_days_in_week
year_grid_height = 1 * nb_rows_per_year

def get_grid_coords_xyz(row, year_min, year_grid_height):
    year_nb = row["Year"] - year_min
    year_grid_x = 0
    year_grid_y = year_grid_height * year_nb
    month = row["Month"] - 1
    month_grid_x, month_grid_y = month//nb_rows_per_year, month - nb_cols_per_year*(month//nb_cols_per_year)
    month_grid_width = nb_days_in_week
    month_grid_height = row['Days_in_month']// nb_days_in_week
    day_grid_x, day_grid_y = row['Day_of_week'], row['Week_of_month']
    day_grid_y = (2*row['Day'] - day_grid_x - 1)//7 + 1

    x = year_grid_x + month_grid_x * month_grid_width  + (day_grid_x)
    y = year_grid_y + month_grid_y * month_grid_height + (day_grid_y)

    return x,y

everyday_grouped_df["grid_coords_xy"] = everyday_grouped_df.apply(get_grid_coords_xyz, args=(year_min, year_grid_height), axis=1)
everyday_grouped_df.head(15)

,Date_yy-mm-dd,Year,Month,Month_name,Day,Day_name,Day_of_week,Week_of_month,Days_in_month,Activity Type,Distance,Calories,Time,Avg HR,Max HR,Total_nb_activities_per_day,grid_coords_xy
0,2021-01-01,2021,1,January,1,Friday,4,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(4, 0)"
1,2021-01-02,2021,1,January,2,Saturday,5,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(5, 0)"
2,2021-01-03,2021,1,January,3,Sunday,6,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(6, 0)"
3,2021-01-04,2021,1,January,4,Monday,0,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(0, 2)"
4,2021-01-05,2021,1,January,5,Tuesday,1,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(1, 2)"
5,2021-01-06,2021,1,January,6,Wednesday,2,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(2, 2)"
6,2021-01-07,2021,1,January,7,Thursday,3,1,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(3, 2)"
7,2021-01-08,2021,1,January,8,Friday,4,2,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(4, 2)"
8,2021-01-09,2021,1,January,9,Saturday,5,2,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(5, 2)"
9,2021-01-10,2021,1,January,10,Sunday,6,2,31,NaN,NaN,NaN,NaN,NaN,NaN,1,"(6, 2)"


In [248]:
9 - ( 5 - 9 + 1)
(2*9 - 5 - 1)//7 + 1

2

## Export dataframes as csv

In [35]:
# set output csv path
out_csv_path = df_path.parent / f"{df_path.stem}_cleaned.csv"
df.to_csv(out_csv_path, index=False)

In [36]:
# set output csv path
out_csv_path = df_path.parent / f"{df_path.stem}_cleaned_grouped.csv"
grouped_df.to_csv(out_csv_path, index=False)

In [250]:
# set output csv path
out_csv_path = df_path.parent / f"{df_path.stem}_cleaned_grouped_all_days.csv"
everyday_grouped_df.to_csv(out_csv_path, index=False)

In [157]:
everyday_grouped_df[(everyday_grouped_df["Year"] == 2021) & (everyday_grouped_df["Month"] == 7)]

,Date_yy-mm-dd,Year,Month,Month_name,Day,Day_name,Day_of_week,Week_of_month,Days_in_month,Activity Type,Distance,Calories,Time,Avg HR,Max HR,Total_nb_activities_per_day
229,2021-07-01,2021,7,July,1,Thursday,3,1,31,Breathwork,0.00,0.0,15.566667,54.0,76.0,3
230,2021-07-01,2021,7,July,1,Thursday,3,1,31,Cycling,24.56,601.0,88.100000,117.0,144.5,3
231,2021-07-01,2021,7,July,1,Thursday,3,1,31,Other,2.63,484.0,94.150000,111.0,131.0,3
232,2021-07-02,2021,7,July,2,Friday,4,1,31,Cycling,24.62,535.0,82.666667,113.0,141.0,2
233,2021-07-02,2021,7,July,2,Friday,4,1,31,Other,3.66,608.0,121.333333,111.0,140.0,2
234,2021-07-03,2021,7,July,3,Saturday,5,1,31,Breathwork,0.00,0.0,1.105000,70.0,96.0,3
235,2021-07-03,2021,7,July,3,Saturday,5,1,31,Cycling,5.01,161.0,20.933333,119.0,152.0,3
236,2021-07-03,2021,7,July,3,Saturday,5,1,31,Other,1.97,316.0,52.050000,113.0,141.0,3
237,2021-07-04,2021,7,July,4,Sunday,6,1,31,Cycling,4.33,125.0,17.825000,110.0,129.0,2
238,2021-07-04,2021,7,July,4,Sunday,6,1,31,Other,6.58,719.0,72.271667,137.5,159.5,2


In [ ]:
# # get list of years (without duplicates) and sort it
# years = everyday_grouped_df["Year"].unique()
# years.sort()

# months = everyday_grouped_df[["Month", "Month_name"]].drop_duplicates()
# months.sort_values(by="Month")

In [ ]:
import bpy
import pandas as pd
import numpy as np

# read csv file into pandas dataframe
df = pd.read_csv("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/data/garmin_data/Activities_20210304_20220818_cleaned_grouped_all_days.csv")

# get list of dates (without duplicates) and sort it
dates = df["Date_yy-mm-dd"].unique()
dates.sort()

# iterate over activity date
for i, activity_date in enumerate(dates):

    # get activities for current date
    activities_times = df.loc[df["Date_yy-mm-dd"] == activity_date, "Time"]
    
    # get day name
    day_name = df.loc[df["Date_yy-mm-dd"] == activity_date, "Day_name"].values[0]
    
    if day_name == "Monday":
        x = 0
    
    elif day_name == "Tuesday":
        x = 2
        
    elif day_name == "Wednesday":
        x = 4
        
    elif day_name == "Thursday":
        x = 6
        
    elif day_name == "Friday":
        x = 8
        
    elif day_name == "Saturday":
        x = 10
        
    elif day_name == "Sunday":
        x = 12
        
    
    # iterate over y values
    for y, time in enumerate(activities_times):
        
        if time == np.nan:
            time = 0.0

        if y%2 == 0:
            x_grid = x
        else:
            x_grid = x + 1
            
        location = (x_grid, y + i*2, time/2)
        
        # add cube
        bpy.ops.mesh.primitive_cube_add(size=1, enter_editmode=False, align='WORLD', location=location, scale=(1, 1, time))

In [ ]:
import bpy
import pandas as pd
import numpy as np

# read csv file into pandas dataframe
df = pd.read_csv("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/data/garmin_data/Activities_20210304_20220818_cleaned_grouped_all_days.csv")

# get list of dates (without duplicates) and sort it
dates = df["Date_yy-mm-dd"].unique()
dates.sort()

x = 0
y = 0
# iterate over activity date
for i, activity_date in enumerate(dates):

    # get activities for current date
    activities_times = df.loc[df["Date_yy-mm-dd"] == activity_date, "Time"]
    
    if i%15 == 0:
        x = 0
        y += 3
        
    else:
        x += 2
    
    # iterate over y values
    for j, time in enumerate(activities_times):
        
        if time == np.nan:
            time = 0.0

        if j%2 == 0:
            x_grid = x
            j += 2
        else:
            x_grid = x + 1
            
        location = (x_grid, y+j, time/2)
        
        # add cube
        bpy.ops.mesh.primitive_cube_add(size=1, enter_editmode=False, align='WORLD', location=location, scale=(1, 1, time))

In [ ]:
import bpy
import pandas as pd
import numpy as np

# read csv file into pandas dataframe
df = pd.read_csv("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/data/garmin_data/Activities_20210304_20220818_cleaned_grouped_all_days.csv")

# get list of dates (without duplicates) and sort it
dates = df["Date_yy-mm-dd"].unique()
dates.sort()

# iterate over activity date
for i, activity_date in enumerate(dates):

    # get activities for current date
    activities_times = df.loc[df["Date_yy-mm-dd"] == activity_date, "Time"]
    xys = df.loc[df["Date_yy-mm-dd"] == activity_date, "grid_coords_xy"]
    
    total_time = 0.0
    
    # iterate over y values
    for j, time in enumerate(activities_times):
        
        if time == np.nan:
            time = 0.0
            
        total_time += time
        
    x, y = xys.iloc[j][1:-1].split(",")
    
    location = (int(x),int(y), time/2)
    
    # add cube
    bpy.ops.mesh.primitive_cube_add(size=1, enter_editmode=False, align='WORLD', location=location, scale=(1, 1, time))